# ⚡ Einstein Benchmark: Laya vs. Rules vs. Einstein

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pchavali09/laya-case-routing-poc/blob/main/notebooks/einstein_benchmark.ipynb)

This notebook trains and evaluates **Laya** and the **Rules Baseline** against the 1,000-case benchmark designed for Salesforce Einstein Case Classification.

- **Training Set:** 800 closed cases (`train_closed.csv`) uploaded to Salesforce to satisfy Einstein data requirements.
- **Test Set:** 200 unseen cases (`test_truth.csv` / `test_blind.csv`) across 4 difficulty tiers: *Easy*, *Paraphrase*, *Multi-Topic*, and *Ambiguous*.
- **Fairness Rule:** Fixed seed `2026`, temperature-calibrated posterior probabilities, zero Einstein hallucination.

In [ ]:
# Step 1: Environment & Setup
import os, sys, subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("🌐 Running inside Google Colab environment...")
    if not os.path.exists("laya-case-routing-poc"):
        print("⏳ Cloning repository https://github.com/pchavali09/laya-case-routing-poc.git...")
        subprocess.run(["git", "clone", "https://github.com/pchavali09/laya-case-routing-poc.git"], check=True)
    os.chdir("laya-case-routing-poc")
    repo_root = os.getcwd()
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch", "pandas", "numpy", "scikit-learn", "matplotlib", "tabulate"])
else:
    repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
    if not os.path.exists(os.path.join(repo_root, "data")):
        repo_root = os.getcwd()

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

from src.rules import classify_case_rules

DEPARTMENTS = ["Billing", "Technical", "Sales", "Operations", "Other"]
DEPT2IDX = {d: i for i, d in enumerate(DEPARTMENTS)}
IDX2DEPT = {i: d for i, d in enumerate(DEPARTMENTS)}
DIFFICULTIES = ["easy", "paraphrase", "multi_topic", "ambiguous"]
THRESHOLDS = [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95]

SEED = 2026
torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"✅ Setup complete! Working directory: {repo_root} | Seed: {SEED}")

In [ ]:
# Step 2: Load Benchmark Datasets
data_dir = os.path.join(repo_root, "data", "einstein_benchmark")
train_df = pd.read_csv(os.path.join(data_dir, "train_closed.csv"))
test_blind_df = pd.read_csv(os.path.join(data_dir, "test_blind.csv"))
test_truth_df = pd.read_csv(os.path.join(data_dir, "test_truth.csv"))

test_df = pd.merge(test_blind_df, test_truth_df, on="ExternalId__c")

print(f"📦 Loaded {len(train_df)} Closed Training Cases across 5 departments (160 each)")
print(f"📦 Loaded {len(test_df)} Unseen Test Cases across 4 difficulty tiers")
print("\nTest difficulty breakdown:")
display(test_df["difficulty"].value_counts().to_frame("Case Count"))

In [ ]:
# Step 3: Define Laya Neural Decision Head
class LayaClassifierHead(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int = 64, num_classes: int = 5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, num_classes)
        )
        self.temperature = nn.Parameter(torch.ones(1) * 1.5)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

    def calibrated_predict_proba(self, x: torch.Tensor):
        with torch.no_grad():
            logits = self.forward(x)
            scaled_logits = logits / torch.clamp(self.temperature, min=0.1, max=10.0)
            probs = torch.softmax(scaled_logits, dim=-1)
            confidences, preds = torch.max(probs, dim=-1)
            return preds, confidences, probs

print("✅ Laya neural classifier head defined.")

In [ ]:
# Step 4: Extract Features & Train Laya on 800 Cases
train_texts = (train_df["Subject"].fillna("") + " " + train_df["Description"].fillna("")).tolist()
test_texts = (test_df["Subject"].fillna("") + " " + test_df["Description"].fillna("")).tolist()

y_train_raw = [DEPT2IDX[d] for d in train_df["Department__c"]]
y_test_raw = [DEPT2IDX[d] for d in test_df["true_department"]]

# 80/20 Stratified Validation split for temperature optimization
train_idx, val_idx = train_test_split(range(len(train_texts)), test_size=0.20, random_state=SEED, stratify=y_train_raw)

vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=512, sublinear_tf=True)
X_train_sub = vectorizer.fit_transform([train_texts[i] for i in train_idx]).toarray()
X_val_sub = vectorizer.transform([train_texts[i] for i in val_idx]).toarray()
X_test = vectorizer.transform(test_texts).toarray()

X_train_t = torch.tensor(X_train_sub, dtype=torch.float32)
y_train_t = torch.tensor([y_train_raw[i] for i in train_idx], dtype=torch.long)
X_val_t = torch.tensor(X_val_sub, dtype=torch.float32)
y_val_t = torch.tensor([y_train_raw[i] for i in val_idx], dtype=torch.long)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test_raw, dtype=torch.long)

model = LayaClassifierHead(input_dim=X_train_sub.shape[1], hidden_dim=64, num_classes=len(DEPARTMENTS))
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.net.parameters(), lr=0.015, weight_decay=1e-4)

print("⚡ Training Laya Classifier...")
for epoch in range(80):
    model.train()
    optimizer.zero_grad()
    logits = model(X_train_t)
    loss = criterion(logits, y_train_t)
    loss.backward()
    optimizer.step()

# Optimize Temperature Parameter T on Validation set
model.eval()
with torch.no_grad():
    val_logits = model(X_val_t)
temp_optimizer = optim.LBFGS([model.temperature], lr=0.01, max_iter=50)
def eval_temp():
    temp_optimizer.zero_grad()
    loss = criterion(val_logits / torch.clamp(model.temperature, min=0.1), y_val_t)
    loss.backward()
    return loss
temp_optimizer.step(eval_temp)
print(f"🔥 Training Complete! Optimal Calibrated Temperature: T = {model.temperature.item():.4f}")

In [ ]:
# Step 5: Accuracy Comparison (Laya vs. Rules)
preds_t, confs_t, probs_t = model.calibrated_predict_proba(X_test_t)
laya_preds = preds_t.numpy()
laya_confs = confs_t.numpy()
y_test_arr = y_test_t.numpy()

# Rules baseline predictions
rules_preds = []
for _, row in test_df.iterrows():
    r_pred, _, _ = classify_case_rules(str(row["Subject"]), str(row["Description"]))
    rules_preds.append(DEPT2IDX[r_pred])
rules_preds = np.array(rules_preds)

laya_correct = (laya_preds == y_test_arr)
rules_correct = (rules_preds == y_test_arr)

results_table = []
results_table.append({
    "Difficulty Tier": "OVERALL (All Cases)",
    "Test Cases": len(test_df),
    "Rules Baseline": f"{rules_correct.sum()}/{len(test_df)} ({rules_correct.mean():.1%})",
    "Laya Classifier": f"{laya_correct.sum()}/{len(test_df)} ({laya_correct.mean():.1%})"
})

for diff in DIFFICULTIES:
    mask = (test_df["difficulty"] == diff).values
    results_table.append({
        "Difficulty Tier": diff,
        "Test Cases": mask.sum(),
        "Rules Baseline": f"{rules_correct[mask].sum()}/{mask.sum()} ({rules_correct[mask].mean():.1%})",
        "Laya Classifier": f"{laya_correct[mask].sum()}/{mask.sum()} ({laya_correct[mask].mean():.1%})"
    })

print("📊 ACCURACY BREAKDOWN BY DIFFICULTY TIER:")
display(pd.DataFrame(results_table))

In [ ]:
# Step 6: Confidence-Gating Table (tau in [0.50, 0.95])
gate_rows = []
for tau in THRESHOLDS:
    routed = (laya_confs >= tau)
    n_routed = routed.sum()
    pct_routed = (n_routed / len(test_df)) * 100.0
    acc_routed = (laya_preds[routed] == y_test_arr[routed]).mean() * 100.0 if n_routed > 0 else np.nan
    gate_rows.append({
        "Threshold (tau)": f"{tau:.2f}",
        "Auto-Routed %": f"{pct_routed:.1f}%",
        "Auto-Routed Accuracy": f"{acc_routed:.1f}%" if not np.isnan(acc_routed) else "N/A",
        "Cases Routed to Human": len(test_df) - n_routed
    })

print("🎯 CONFIDENCE GATING SWEEP:")
display(pd.DataFrame(gate_rows))

In [ ]:
# Step 7: Plot Coverage vs. Accuracy Curve
taus = THRESHOLDS
auto_pcts = [((laya_confs >= t).sum() / len(test_df)) * 100 for t in taus]
auto_accs = [
    (laya_preds[laya_confs >= t] == y_test_arr[laya_confs >= t]).mean() * 100
    for t in taus
]

fig, ax1 = plt.subplots(figsize=(8.5, 4.5))
color = "#1f77b4"
ax1.set_xlabel("Confidence Threshold (tau)", fontsize=11, fontweight="bold")
ax1.set_ylabel("Auto-Routed Cases (%)", color=color, fontsize=11, fontweight="bold")
ax1.plot(taus, auto_pcts, color=color, marker="o", linewidth=2.5, label="Auto-Routed %")
ax1.tick_params(axis="y", labelcolor=color)
ax1.grid(True, linestyle="--", alpha=0.4)

ax2 = ax1.twinx()
color = "#2ca02c"
ax2.set_ylabel("Auto-Routed Accuracy (%)", color=color, fontsize=11, fontweight="bold")
ax2.plot(taus, auto_accs, color=color, marker="s", linewidth=2.5, linestyle="--", label="Accuracy")
ax2.tick_params(axis="y", labelcolor=color)

plt.title("Laya Confidence-Gating Frontier: Auto-Route Coverage vs. Accuracy", fontsize=12, pad=12)
fig.tight_layout()
plt.show()

In [ ]:
# Step 8: Head-to-Head Comparison with Salesforce Einstein
einstein_path = os.path.join(data_dir, "einstein_predictions.csv")
if os.path.exists(einstein_path):
    print("🎉 Einstein predictions found! Loading side-by-side comparison...")
    einstein_df = pd.read_csv(einstein_path)
    merged = test_df.merge(pd.DataFrame({
        "ExternalId__c": test_df["ExternalId__c"],
        "Laya_Dept": [IDX2DEPT[p] for p in laya_preds],
        "Laya_Conf": laya_confs
    }), on="ExternalId__c").merge(einstein_df, on="ExternalId__c")
    
    merged["einstein_correct"] = (merged["Einstein_Department"] == merged["true_department"])
    merged["laya_correct"] = (merged["Laya_Dept"] == merged["true_department"])
    
    comp_table = pd.DataFrame([{
        "Model": "Salesforce Einstein",
        "Accuracy": f"{merged['einstein_correct'].mean():.1%}"
    }, {
        "Model": "Laya Classifier",
        "Accuracy": f"{merged['laya_correct'].mean():.1%}"
    }])
    display(comp_table)
else:
    print("=" * 65)
    print("⏳ WAITING FOR EINSTEIN PREDICTIONS FROM YOUR SALESFORCE ORG")
    print("=" * 65)
    print(f"File to create: {einstein_path}")
    print("Columns needed: ExternalId__c, Einstein_Department, Einstein_Confidence")
    print("Once placed here, rerun this cell to view the head-to-head table!")